# 1D AMS reweighting

Double well: `V_theta(x) = theta (x^2 - 1)^2`. Reference AMS is run once at `theta_ref`; four direct AMS runs are added as comparison points.


In [ ]:
from pathlib import Path
import shutil

import matplotlib.pyplot as plt
import numpy as np
import ase.units as units
from ase import Atoms
from ase.calculators.calculator import Calculator, all_changes
from ase.io import read, write
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from aseams.ams import AMS
from aseams.cvs import CollectiveVariables

from girsanov_uq.integrators import LangevinOBABO
from girsanov_uq.post_processing.reweighting import Reweightor


In [ ]:
theta_ref = 0.20
direct_theta_values = theta_ref + np.array([-0.06, -0.03, 0.03, 0.06])

temperature_K = 1800.0
dt = 1.0 * units.fs
friction = 0.01 / units.fs
mass = 1.0

n_replicas = 50
r_boundary = -0.8
p_boundary = 0.6
max_exit_steps = 20_000
max_length_iter = 8_000

workdir = Path("simple_ams_reweighting_1d_run")
ini_dir = workdir / "ini_conds"
ams_dir = workdir / "ams"


In [ ]:
class DoubleWellCalculator(Calculator):
    implemented_properties = ["energy", "forces", "stress", "descriptors", "grad_descriptors"]

    def __init__(self, theta, **kwargs):
        super().__init__(**kwargs)
        self.theta_vec = np.array([float(theta)])

    @staticmethod
    def descriptor(x):
        return (x**2 - 1.0) ** 2

    @staticmethod
    def descriptor_grad(x):
        return 4.0 * x * (x**2 - 1.0)

    def calculate(self, atoms=None, properties=None, system_changes=all_changes):
        super().calculate(atoms, properties, system_changes)
        x = float(self.atoms.positions[0, 0])
        d = self.descriptor(x)
        ddx = self.descriptor_grad(x)

        forces = np.zeros((len(self.atoms), 3))
        forces[0, 0] = -self.theta_vec[0] * ddx

        grad = np.zeros((len(self.atoms), 1, 3))
        grad[0, 0, 0] = ddx

        self.results.update(
            energy=float(self.theta_vec[0] * d),
            forces=forces,
            stress=np.zeros(6),
            descriptors=np.array([d]),
            grad_descriptors=grad,
        )

    def get_descriptors(self, atoms):
        self.calculate(atoms, properties=["descriptors"])
        return self.results["descriptors"]

    def get_descriptors_jacobian(self, atoms):
        self.calculate(atoms, properties=["grad_descriptors"])
        return self.results["grad_descriptors"]


## Setup

Reactant: `x <= -0.8`; product: `x >= 0.6`; reaction coordinate: `x`.


In [ ]:
def x(atoms):
    return float(atoms.positions[0, 0])

cv = CollectiveVariables(cv_r=x, cv_p=x, reaction_coordinate=x)
cv.set_r_crit("below")
cv.set_in_r_boundary(float(r_boundary))
cv.set_sigma_r_level(float(r_boundary))
cv.set_out_of_r_zone(float(r_boundary))
cv.set_p_crit("above")
cv.set_in_p_boundary(float(p_boundary))


def make_atoms(theta, x0=-1.0):
    atoms = Atoms("H", positions=[[x0, 0.0, 0.0]], cell=(20.0, 20.0, 20.0))
    atoms.set_masses([mass])
    atoms.calc = DoubleWellCalculator(theta)
    return atoms


def make_dyn(atoms, rng, record_noise):
    MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K, rng=rng)
    return LangevinOBABO(
        atoms,
        timestep=dt,
        temperature_K=temperature_K,
        friction=friction,
        fixcm=False,
        rng=rng,
        record_noise=record_noise,
        logfile=None,
        trajectory=None,
    )


def sample_exit(theta, seed):
    rng = np.random.default_rng(seed)
    atoms = make_atoms(theta, x0=-1.0 + 0.03 * rng.normal())
    dyn = make_dyn(atoms, rng, record_noise=True)
    for _ in range(max_exit_steps):
        dyn.run(1)
        if x(atoms) > r_boundary:
            atoms.info.clear()
            atoms.info["weight_ini_cond"] = 1.0
            return atoms.copy()
    raise RuntimeError("Initial condition did not leave R.")


def write_initial_conditions(theta, folder, seed0):
    folder.mkdir(parents=True, exist_ok=True)
    for i in range(n_replicas):
        write(folder / f"ini_{i}.extxyz", sample_exit(theta, seed0 + i), format="extxyz")


def run_ams(theta, run_dir, seed0, record_noise=False):
    if run_dir.exists():
        shutil.rmtree(run_dir)
    run_ini = run_dir / "ini_conds"
    run_ams = run_dir / "ams"
    run_ams.mkdir(parents=True)
    write_initial_conditions(theta, run_ini, seed0)

    atoms = make_atoms(theta, x0=-0.7)
    dyn = make_dyn(atoms, np.random.default_rng(seed0 + 100_000), record_noise=record_noise)
    ams = AMS(
        n_rep=n_replicas,
        k_min=1,
        dyn=dyn,
        xi=cv,
        fixcm=False,
        save_all=False,
        rc_threshold=1e-8,
        max_length_iter=max_length_iter,
        verbose=False,
        rng=np.random.default_rng(seed0 + 200_000),
    )
    ams.set_ini_cond_dir(str(run_ini))
    ams.set_ams_dir(str(run_ams), clean=False)
    ams.run()
    return ams, run_ams


In [ ]:
grid = np.linspace(-1.5, 1.5, 400)
fig, ax = plt.subplots(figsize=(6, 3.5))
for theta in [0.14, theta_ref, 0.26]:
    ax.plot(grid, theta * (grid**2 - 1.0) ** 2, label=f"theta={theta:.2f}")
ax.axvspan(-1.5, r_boundary, color="tab:blue", alpha=0.12, label="R")
ax.axvspan(p_boundary, 1.5, color="tab:orange", alpha=0.12, label="P")
ax.set(xlabel="x", ylabel="V_theta(x) [eV]", title="1D double well")
ax.legend()
plt.show()


## Reference run and reweighting


In [ ]:
if workdir.exists():
    shutil.rmtree(workdir)

ams_ref, ams_dir = run_ams(theta_ref, workdir / "reference", seed0=10_000, record_noise=True)
p_ref = ams_ref.p_ams()
print(f"p(theta_ref={theta_ref:.2f}) = {p_ref:.6g}")
print(f"success={ams_ref.success}, iterations={ams_ref.ams_it}")


In [ ]:
rw = Reweightor(
    calculator=DoubleWellCalculator(theta_ref),
    mode="linear",
    temperature_K=temperature_K,
    dt=dt,
    friction=friction,
    masses=np.array([mass]),
)

records = []
for i in range(n_replicas):
    traj = read(ams_dir / f"rep_{i}.traj", index=":")
    if traj and cv.in_p(traj[-1]):
        score, fim = rw.reweight_one_trajectory_linear(traj)
        records.append({
            "weight": float(traj[-1].info.get("replica_weight", ams_ref.rep_weights[i])),
            "score": float(np.ravel(score)[0]),
            "fim": float(np.ravel(fim)[0]),
        })

len(records), records[:3]


In [ ]:
def p_reweighted(theta_values):
    theta_values = np.asarray(theta_values, dtype=float)
    p = np.zeros_like(theta_values)
    for r in records:
        dtheta = theta_values - theta_ref
        p += r["weight"] * np.exp(dtheta * r["score"] - 0.5 * dtheta**2 * r["fim"])
    return p


theta_grid = np.linspace(theta_ref - 0.12, theta_ref + 0.12, 121)
p_grid = p_reweighted(theta_grid)

direct = []
for j, theta in enumerate(direct_theta_values):
    ams, _ = run_ams(float(theta), workdir / f"direct_{theta:.3f}", seed0=30_000 + 1_000 * j)
    direct.append({"theta": float(theta), "p": float(ams.p_ams()), "success": ams.success})
    print(direct[-1])

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(theta_grid, p_grid, lw=2, label="reweighted from theta_ref")
ax.scatter([theta_ref], [p_ref], color="black", zorder=3, label="reference AMS")
ax.scatter([r["theta"] for r in direct], [r["p"] for r in direct], marker="s", color="tab:red", zorder=4, label="direct AMS")
ax.set(xlabel="theta [eV]", ylabel="transition probability", title="AMS reweighting check")
ax.legend()
plt.show()


With few replicas the direct AMS points are noisy. Increase `n_replicas` and repeat seeds for production.
